# Ручной подбор параметров AR/MA/ARMA/ARIMA/SARIMAX

Этот блокнот оставлен как расширенный исследовательский вариант. Основной учебный путь находится в `training_pipeline_auto_arima_stock_forecasting.ipynb`, где параметры подбирает `auto_arima`.

Здесь показана более подробная идея: мы сами задаём набор параметров, обучаем несколько моделей и сравниваем их на контрольном отрезке.

In [ ]:
import itertools
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

TICKERS = ["AAPL", "MSFT", "NVDA"]
PERIOD = "5y"
TEST_SIZE = 90
SEASONAL_PERIOD = 5
MAXITER = 60

## 1. Загрузка данных

Берём цену закрытия `Close`, приводим индекс к датам и заполняем пропуски по рабочим дням.

In [ ]:
def load_close_prices(ticker):
    data = yf.download(
        ticker,
        period=PERIOD,
        interval="1d",
        auto_adjust=True,
        progress=False,
        threads=False,
    )

    if isinstance(data.columns, pd.MultiIndex):
        close = data[("Close", ticker)]
    else:
        close = data["Close"]

    close = close.dropna().astype(float)
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.asfreq("B").ffill().dropna()

example_close = load_close_prices("AAPL")
example_close.tail()

In [ ]:
example_close.plot(title="Цена закрытия AAPL")
plt.xlabel("Дата")
plt.ylabel("Цена закрытия, доллары США")
plt.show()

## 2. Проверка стационарности

Для исходного ряда и первой разности считаем ADF-тест.

In [ ]:
adf_original = adfuller(example_close.dropna())
adf_diff = adfuller(example_close.diff().dropna())

print("Исходный ряд")
print(f"ADF statistic: {adf_original[0]:.4f}")
print(f"p-value: {adf_original[1]:.4f}")
print()
print("Первая разность")
print(f"ADF statistic: {adf_diff[0]:.4f}")
print(f"p-value: {adf_diff[1]:.4f}")

## 3. Набор моделей для ручной проверки

Параметры задаются явно:

- `AR(p)` как `(p, 0, 0)`;
- `MA(q)` как `(0, 0, q)`;
- `ARMA(p, q)` как `(p, 0, q)`;
- `ARIMA(p, d, q)`;
- `SARIMAX(p, d, q)(P, D, Q, s)`.

In [ ]:
model_candidates = []

for p in range(1, 4):
    model_candidates.append({"family": "AR", "order": (p, 0, 0), "seasonal_order": (0, 0, 0, 0)})

for q in range(1, 4):
    model_candidates.append({"family": "MA", "order": (0, 0, q), "seasonal_order": (0, 0, 0, 0)})

for p, q in itertools.product(range(1, 3), range(1, 3)):
    model_candidates.append({"family": "ARMA", "order": (p, 0, q), "seasonal_order": (0, 0, 0, 0)})

for d in (1, 2):
    for p, q in itertools.product(range(0, 3), range(0, 3)):
        if p != 0 or q != 0:
            model_candidates.append({"family": "ARIMA", "order": (p, d, q), "seasonal_order": (0, 0, 0, 0)})

for p, q in itertools.product(range(0, 2), range(0, 2)):
    for P, D, Q in itertools.product(range(0, 2), range(0, 2), range(0, 2)):
        if P == 0 and D == 0 and Q == 0:
            continue
        if p == 0 and q == 0 and P == 0 and Q == 0:
            continue
        model_candidates.append({
            "family": "SARIMAX",
            "order": (p, 1, q),
            "seasonal_order": (P, D, Q, SEASONAL_PERIOD),
        })

print("Количество вариантов:", len(model_candidates))
pd.DataFrame(model_candidates).head()

## 4. Обучение и сравнение

Модели обучаются на масштабированном ряде, а ошибки считаются уже в исходном масштабе цен.

In [ ]:
all_results = []

for ticker in TICKERS:
    print("=" * 80)
    print("Биржевой символ:", ticker)

    close = load_close_prices(ticker)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(close.to_numpy().reshape(-1, 1)).ravel()

    train = scaled[:-TEST_SIZE]
    test = scaled[-TEST_SIZE:]
    test_prices = scaler.inverse_transform(test.reshape(-1, 1)).ravel()

    for candidate in model_candidates:
        row = {
            "ticker": ticker,
            "family": candidate["family"],
            "order": candidate["order"],
            "seasonal_order": candidate["seasonal_order"],
        }
        try:
            model = SARIMAX(
                train,
                order=candidate["order"],
                seasonal_order=candidate["seasonal_order"],
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            result = model.fit(disp=False, maxiter=MAXITER)
            forecast_scaled = np.asarray(result.forecast(steps=len(test)), dtype=float)
            forecast_prices = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).ravel()
            forecast_prices = np.maximum(forecast_prices, 0)

            row["status"] = "ok"
            row["mae"] = mean_absolute_error(test_prices, forecast_prices)
            row["rmse"] = np.sqrt(mean_squared_error(test_prices, forecast_prices))
            row["mape"] = np.mean(np.abs((test_prices - forecast_prices) / test_prices)) * 100
            row["aic"] = result.aic
            row["bic"] = result.bic
        except Exception as error:
            row["status"] = "failed"
            row["error"] = str(error)
            row["mae"] = np.nan
            row["rmse"] = np.nan
            row["mape"] = np.nan
            row["aic"] = np.nan
            row["bic"] = np.nan

        all_results.append(row)

results_df = pd.DataFrame(all_results)
results_df.to_csv(MODEL_DIR / "model_results.csv", index=False)
results_df.head()

## 5. Лучшие модели по контрольному отрезку

In [ ]:
best_by_ticker = (
    results_df[results_df["status"] == "ok"]
    .sort_values(["ticker", "rmse", "mape", "aic"])
    .groupby("ticker")
    .head(1)
)

best_by_ticker[["ticker", "family", "order", "seasonal_order", "mae", "rmse", "mape", "aic", "bic"]]

In [ ]:
family_comparison = (
    results_df[results_df["status"] == "ok"]
    .groupby("family")[["mae", "rmse", "mape", "aic", "bic"]]
    .mean()
    .sort_values("rmse")
)
family_comparison

## Итог

Этот блокнот показывает ручной подбор параметров. Для основного приложения удобнее использовать `training_pipeline_auto_arima_stock_forecasting.ipynb`, потому что там параметры подбирает `auto_arima`, как в теоретическом материале.